In [12]:
# _*_ coding:utf-8 _*_

from tqdm import tqdm
from scipy import optimize
import tensorflow as tf
from utils import *
import json
import os

seed = 12345
np.random.seed(seed)

#choose the GPU, "-1" represents using the CPU
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

## Helpers for downloading zipped embeddings folder

In [13]:
import os
import tempfile
import zipfile

# try to import requests, otherwise we'll use urllib
try:
    import requests
    HAS_REQUESTS = True
except Exception:
    import urllib.request
    HAS_REQUESTS = False


def download_and_extract_file(url, inner_path, dest_path=None, chunk_size=8192):
    """
    Download a zip from `url`, open it, and extract the file `inner_path` (path inside the zip).
    If dest_path is:
      - None: returns the file bytes.
      - a directory: writes the inner file into that directory using its basename.
      - a file path: writes to that exact file path.
    Returns the bytes if dest_path is None, otherwise returns the path to the written file.

    Raises:
      - FileNotFoundError if inner_path not found in the zip
      - RuntimeError for download / extraction problems
    """
    # 1) Download to a temporary file
    tmp_fd, tmp_path = tempfile.mkstemp(suffix=".zip")
    os.close(tmp_fd)  # we'll open it normally
    try:
        if HAS_REQUESTS:
            with requests.get(url, stream=True, timeout=30) as r:
                r.raise_for_status()
                with open(tmp_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=chunk_size):
                        if chunk:
                            f.write(chunk)
        else:
            # urllib fallback
            with urllib.request.urlopen(url, timeout=30) as resp, open(tmp_path, "wb") as f:
                while True:
                    chunk = resp.read(chunk_size)
                    if not chunk:
                        break
                    f.write(chunk)

        # 2) Open the zip and extract single file safely
        with zipfile.ZipFile(tmp_path, "r") as z:
            # normalize names inside zip
            namelist = z.namelist()
            if inner_path not in namelist:
                # allow matching by basename if exact path not found
                matches = [n for n in namelist if os.path.basename(n) == os.path.basename(inner_path)]
                if len(matches) == 1:
                    inner_path = matches[0]
                elif len(matches) > 1:
                    raise FileNotFoundError(
                        f"Ambiguous inner filename '{inner_path}'. Multiple matches: {matches}"
                    )
                else:
                    raise FileNotFoundError(f"'{inner_path}' not found in zip (checked {len(namelist)} entries).")

            # read bytes from the zip (avoid zip-slip because we do not extract paths)
            with z.open(inner_path, "r") as inner_file:
                data = inner_file.read()

            if dest_path is None:
                return data

            # decide final output path
            if os.path.isdir(dest_path):
                out_path = os.path.join(dest_path, os.path.basename(inner_path))
            else:
                # if dest_path endswith a path separator treat as dir
                if dest_path.endswith(os.path.sep):
                    os.makedirs(dest_path, exist_ok=True)
                    out_path = os.path.join(dest_path, os.path.basename(inner_path))
                else:
                    out_path = dest_path
                    out_dir = os.path.dirname(out_path)
                    if out_dir:
                        os.makedirs(out_dir, exist_ok=True)

            # write file atomically
            tmp_out_fd, tmp_out_path = tempfile.mkstemp(dir=os.path.dirname(out_path))
            try:
                with os.fdopen(tmp_out_fd, "wb") as wf:
                    wf.write(data)
                os.replace(tmp_out_path, out_path)
            finally:
                if os.path.exists(tmp_out_path):
                    os.remove(tmp_out_path)

            return out_path

    finally:
        # clean up downloaded zip
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

## Load embeddings from zipped folder

In [14]:
out_path = download_and_extract_file(
    "http://nlp.stanford.edu/data/glove.6B.zip",
    "glove.6B.300d.txt",
    dest_path="./glove.6B.300d.txt"
)

In [15]:
#load the pre-trained word embeddings
#please download the zip file from "http://nlp.stanford.edu/data/glove.6B.zip" and choose "glove.6B.300d.txt" as the word vectors.

word_vecs = {}
with open("./glove.6B.300d.txt",encoding='UTF-8') as f:
    for line in tqdm(f.readlines()):
        line = line.split()
        word_vecs[line[0]] = np.array([float(x) for x in line[1:]])

100%|██████████| 400000/400000 [00:14<00:00, 27185.61it/s]


# Testing

In [16]:
#load the translated entity names 

ent_names = json.load(open("translated_ent_name/dbp_ja_en.json","r"))

#load KGs and test set

file_path = "KGs/dbp_ja_en/"
all_triples,node_size,rel_size = load_triples(file_path,True)
train_pair,test_pair = load_aligned_pair(file_path,ratio=0)

In [17]:
#generate the bigram dictionary

d = {}
count = 0
for _,name in ent_names:
    for word in name:
        word = word.lower()
        for idx in range(len(word)-1):
            if word[idx:idx+2] not in d:
                d[word[idx:idx+2]] = count
                count += 1

In [18]:
#generate the word-level features and char-level features

ent_vec = np.zeros((node_size,300))
char_vec = np.zeros((node_size,len(d)))
for i,name in ent_names:
    k = 0
    for word in name:
        word = word.lower()
        if word in word_vecs:
            ent_vec[i] += word_vecs[word]
            k += 1
        for idx in range(len(word)-1):
            char_vec[i,d[word[idx:idx+2]]] += 1
    if k:
        ent_vec[i]/=k
    else:
        ent_vec[i] = np.random.random(300)-0.5
        
    if np.sum(char_vec[i]) == 0:
        char_vec[i] = np.random.random(len(d))-0.5
    ent_vec[i] = ent_vec[i]/ np.linalg.norm(ent_vec[i])
    char_vec[i] = char_vec[i]/ np.linalg.norm(char_vec[i])

In [19]:
#build the relational adjacency matrix

dr = {}
for x,r,y in all_triples:
    if r not in dr:
        dr[r] = 0
    dr[r] += 1
    
sparse_rel_matrix = []
for i in range(node_size):
    sparse_rel_matrix.append([i,i,np.log(len(all_triples)/node_size)]);
for h,r,t in all_triples:
    sparse_rel_matrix.append([h,t,np.log(len(all_triples)/dr[r])])

sparse_rel_matrix = np.array(sorted(sparse_rel_matrix,key=lambda x:x[0]))
sparse_rel_matrix = tf.SparseTensor(indices=sparse_rel_matrix[:,:2],values=sparse_rel_matrix[:,2],dense_shape=(node_size,node_size))

In [20]:
#feature selection 

mode = "hybrid-level"

if mode == "word-level":
    feature = ent_vec
if mode == "char-level":
    feature = char_vec
if mode == "hybrid-level": 
    feature = np.concatenate([ent_vec,char_vec],-1)
feature = tf.nn.l2_normalize(feature,axis=-1)

In [21]:
%%time
#choose the graph depth L and feature propagation

depth = 2
def cal_sims(test_pair,feature):
    feature_a = tf.gather(indices=test_pair[:,0],params=feature)
    feature_b = tf.gather(indices=test_pair[:,1],params=feature)
    return tf.matmul(feature_a,tf.transpose(feature_b,[1,0]))

sims = cal_sims(test_pair,feature)
for i in range(depth):    
    feature = tf.sparse.sparse_dense_matmul(sparse_rel_matrix,feature)
    feature = tf.nn.l2_normalize(feature,axis=-1)
    sims += cal_sims(test_pair,feature)
sims /= depth+1

CPU times: total: 3min 34s
Wall time: 7.69 s


In [22]:
%%time
#solving by Hungarian algorithm, only for the CPU
result = optimize.linear_sum_assignment(sims,maximize=True)
test(result,"hungarian")

hits@1 : 96.32%
CPU times: total: 12.8 s
Wall time: 13 s


In [23]:
%%time
#solving by Sinkhorn operation

sims = tf.exp(sims*50)
for k in range(10):
    sims = sims / tf.reduce_sum(sims,axis=1,keepdims=True)
    sims = sims / tf.reduce_sum(sims,axis=0,keepdims=True)
test(sims,"sinkhorn")

hits@1 : 95.81% hits@10 : 98.85% MRR : 96.97%
CPU times: total: 1min 27s
Wall time: 3.2 s
